In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import anndata
import scanpy as sc

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
clusters = pd.read_csv(f'{projdir}/csv/integrate_m3c/leiden_clusters.csv', sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
tmp_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
tmp_clusters = pd.read_csv(tmp_clusters_path, sep='\t', index_col=0)
tmp_clusters.shape

In [ ]:
clusters['snmct_cluster'] = tmp_clusters.reindex(clusters.index)['pred_coarse']
clusters['snmct_cluster'] = pd.Categorical(clusters['snmct_cluster'], categories=tmp_order, ordered=True)
clusters['leiden'] = clusters['leiden'].astype(str)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_clusters = clusters[clusters['line'] == d]
    sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='leiden', s=5, ax=ax)
    ax.set_title(d)
    ax.legend_.remove()
plt.subplots_adjust(wspace=0.4, hspace=0.4)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_clusters = clusters[clusters['line'] == d].copy()
    tmp_clusters['snmct_cluster'] = pd.Categorical(tmp_clusters['snmct_cluster'], categories=tmp_order, ordered=True)
    sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='snmct_cluster', s=5, ax=ax)
    ax.set_title(d)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
#     ax.legend_.remove()
plt.subplots_adjust(wspace=0.5, hspace=0.5)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    tmp_clusters = clusters[clusters['line'] == d].copy()
    tmp_clusters['snmct_cluster'] = pd.Categorical(tmp_clusters['snmct_cluster'], categories=tmp_order, ordered=True)
    sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='snmct_cluster', s=5, ax=ax)
    ax.set_title(d)
#     sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    ax.legend_.remove()
plt.subplots_adjust(wspace=0.5, hspace=0.5)

### include 3C for correct annotation

In [ ]:
indir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/snm3C/higashi'
adata = sc.read_h5ad(f'{indir}/h5ad/embedding/higashi_filtered.h5ad')

In [ ]:
adata

In [ ]:
indir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/snmCT/mc'

tmp_clusters = pd.read_csv(f'{indir}/csv/label_transfer/xgboost_time_v7.csv', sep='\t', index_col=0)
tmp_clusters.shape

In [ ]:
adata.obs['cluster'] = tmp_clusters.reindex(adata.obs.index)['cluster']
adata.obs['cluster_sub'] = tmp_clusters.reindex(adata.obs.index)['cluster_sub']

In [ ]:
sc.pl.umap(adata, color=['cluster'])

### do it donor by donor

In [ ]:
clust_results = []

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
i = 0
d = donors[i]
tmp_clusters = clusters[clusters['line'] == d].copy()
tmp_clusters['snmct_cluster'] = pd.Categorical(tmp_clusters['snmct_cluster'], categories=tmp_order, ordered=True)

###
ax = axes[0]
sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='snmct_cluster', s=10, ax=ax)
ax.set_title(d)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

ax = axes[1]
sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='leiden', s=10, ax=ax)
ax.set_title(d)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.subplots_adjust(wspace=0.8)

In [ ]:
c29_dict = {'4' : 'IPS_1', '0' : 'IPS_2', 
            '1' : 'Fibro_1', '2' : 'Fibro_2', '3' : 'Fibro_3', 
             '5' : 'Fibro_4', '8' : 'Fibro_5', '9' : 'Fibro_6',
            '7' : 'Fibro_7', '10' : 'Fibro_8',
           '6' : 'Inter_1'}

In [ ]:
tmp_clusters['tmp_clust'] = [c29_dict.get(x) for x in tmp_clusters['leiden']]

ax = sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='tmp_clust', s=10)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.subplots_adjust(wspace=0.8)

select = ['leiden', 'tmp_clust']
clust_results.append(tmp_clusters[select])

In [ ]:
d = donors[i]
donor_adata = adata[adata.obs['line'] == d]
donor_adata.obs['renamed_leiden'] = tmp_clusters.reindex(donor_adata.obs.index)['tmp_clust']
sc.pl.umap(donor_adata, color=['renamed_leiden'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
i = 1
d = donors[i]
tmp_clusters = clusters[clusters['line'] == d].copy()
tmp_clusters['snmct_cluster'] = pd.Categorical(tmp_clusters['snmct_cluster'], categories=tmp_order, ordered=True)

###
ax = axes[0]
sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='snmct_cluster', s=10, ax=ax)
ax.set_title(d)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

ax = axes[1]
sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='leiden', s=10, ax=ax)
ax.set_title(d)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.subplots_adjust(wspace=0.8)

In [ ]:
c37_dict = {'7' : 'IPS_1',
            '0' : 'Fibro_1', '1' : 'Fibro_2', '2' : 'Fibro_3', 
             '3' : 'Fibro_4', '5' : 'Fibro_5', '6' : 'Fibro_6',
            '8' : 'Fibro_7',
           '4' : 'Inter_1'}

In [ ]:
tmp_clusters['tmp_clust'] = [c37_dict.get(x) for x in tmp_clusters['leiden']]

ax = sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='tmp_clust', s=10)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.subplots_adjust(wspace=0.8)

select = ['leiden', 'tmp_clust']
clust_results.append(tmp_clusters[select])

In [ ]:
d = donors[i]
donor_adata = adata[adata.obs['line'] == d]
donor_adata.obs['renamed_leiden'] = tmp_clusters.reindex(donor_adata.obs.index)['tmp_clust']
sc.pl.umap(donor_adata, color=['renamed_leiden'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
i = 2
d = donors[i]
tmp_clusters = clusters[clusters['line'] == d].copy()
tmp_clusters['snmct_cluster'] = pd.Categorical(tmp_clusters['snmct_cluster'], categories=tmp_order, ordered=True)

###
ax = axes[0]
sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='snmct_cluster', s=10, ax=ax)
ax.set_title(d)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

ax = axes[1]
sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='leiden', s=10, ax=ax)
ax.set_title(d)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.subplots_adjust(wspace=0.8)

In [ ]:
c38_dict = {'3' : 'IPS_1',
            '0' : 'Fibro_1', '1' : 'Fibro_2', '2' : 'Fibro_3', 
             '4' : 'Fibro_4', '6' : 'Fibro_5', '7' : 'Fibro_6',
            '8' : 'Fibro_7', '9' : 'Fibro_8', '10' : 'Fibro_9',
           '5' : 'Inter_1'}

In [ ]:
tmp_clusters['tmp_clust'] = [c38_dict.get(x) for x in tmp_clusters['leiden']]

ax = sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='tmp_clust', s=10)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.subplots_adjust(wspace=0.8)

select = ['leiden', 'tmp_clust']
clust_results.append(tmp_clusters[select])

In [ ]:
d = donors[i]
donor_adata = adata[adata.obs['line'] == d]
donor_adata.obs['renamed_leiden'] = tmp_clusters.reindex(donor_adata.obs.index)['tmp_clust']
sc.pl.umap(donor_adata, color=['renamed_leiden'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
i = 3
d = donors[i]
tmp_clusters = clusters[clusters['line'] == d].copy()
tmp_clusters['snmct_cluster'] = pd.Categorical(tmp_clusters['snmct_cluster'], categories=tmp_order, ordered=True)

###
ax = axes[0]
sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='snmct_cluster', s=20, ax=ax)
ax.set_title(d)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

ax = axes[1]
sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='leiden', s=20, ax=ax)
ax.set_title(d)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.subplots_adjust(wspace=0.8)

In [ ]:
# ### og
# c39_dict = {'1' : 'IPS_1', '4' : 'IPS_2',
#             '0' : 'Fibro_1', '2' : 'Fibro_2', '5' : 'Fibro_3', 
#              '7' : 'Fibro_4', '8' : 'Fibro_5', '9' : 'Fibro_6',
#             '10' : 'Fibro_7', '11' : 'Fibro_8', '12' : 'Fibro_9',
#             '13' : 'Fibro_10', '14' : 'Fibro_11', '15' : 'Fibro_12',
#             '16' : 'Fibro_13', '17' : 'Fibro_14',
#            '3' : 'Inter_1', '6' : 'Inter_2'}

In [ ]:
c39_dict = {'4' : 'IPS_1', '1' : 'IPS_2',
            '0' : 'Fibro_1', '2' : 'Fibro_2', '5' : 'Fibro_3', 
             '7' : 'Fibro_4', '8' : 'Fibro_5', '9' : 'Fibro_6',
            '10' : 'Fibro_7', '11' : 'Fibro_8', '12' : 'Fibro_9',
            '13' : 'Fibro_10', '14' : 'Fibro_11', '15' : 'Fibro_12',
            '16' : 'Fibro_13', '17' : 'Fibro_14',
           '3' : 'Inter_1', '6' : 'Inter_2'}

In [ ]:
tmp_clusters['tmp_clust'] = [c39_dict.get(x) for x in tmp_clusters['leiden']]

ax = sns.scatterplot(data=tmp_clusters, x='umap0', y='umap1', hue='tmp_clust', s=10)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.subplots_adjust(wspace=0.8)

select = ['leiden', 'tmp_clust']
clust_results.append(tmp_clusters[select])

In [ ]:
d = donors[i]
donor_adata = adata[adata.obs['line'] == d]
donor_adata.obs['renamed_leiden'] = tmp_clusters.reindex(donor_adata.obs.index)['tmp_clust']
sc.pl.umap(donor_adata, color=['renamed_leiden'])

### merge

In [ ]:
tmp_clusts = pd.concat(clust_results)
clusters['renamed_leiden'] = tmp_clusts.reindex(clusters.index)['tmp_clust']
clusters['renamed_leiden'].value_counts()

In [ ]:
replace_dict = {'IPS_1' : 'IPS_alt', 'IPS_2' : 'IPS_main'}
clusters['final_leiden'] = clusters['renamed_leiden'].replace(replace_dict)

In [ ]:
for d in donors:
    donor_clusters = clusters[clusters['line'] == d]
    print(donor_clusters['final_leiden'].value_counts())

In [ ]:
for i in range(4):
    d = donors[i]
    donor_adata = adata[adata.obs['line'] == d]
    donor_adata.obs['final_leiden'] = clusters.reindex(donor_adata.obs.index)['final_leiden']
    sc.pl.umap(donor_adata, color=['final_leiden'], title=d)

In [ ]:
for d in donors:
    donor_clusters = clusters[clusters['line'] == d]
    tmp_clusters = donor_clusters[donor_clusters['final_leiden'] == 'IPS_main']
    print(tmp_clusters['time'].value_counts().sort_index() / tmp_clusters['time'].value_counts().sum())

In [ ]:
for d in donors:
    donor_clusters = clusters[clusters['line'] == d]
    tmp_clusters = donor_clusters[donor_clusters['final_leiden'] == 'IPS_alt']
    print(tmp_clusters['time'].value_counts().sort_index() / tmp_clusters['time'].value_counts().sum())

In [ ]:
clusters.to_csv(f'{projdir}/csv/integrate_m3c/leiden_clusters_renamed.csv', sep='\t')

In [ ]:
!date